# LSTM Uncertainty Estimation Validation

Notebook ini memvalidasi uncertainty estimation berbasis Monte Carlo Dropout untuk checkpoint LSTM aktif. Semua deep learning logic tetap berada di `.ipynb`, sesuai requirement kerja terbaru.

In [1]:
from __future__ import annotations

import json
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "traffic_prediction").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / "src"
if not SRC.exists():
    raise RuntimeError(f"Could not locate project src directory from {Path.cwd()}")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from traffic_prediction.data.scalers import ScalerStore
from traffic_prediction.models.lstm import LSTMModelConfig, TrafficLSTM

MODEL_ROOT = PROJECT_ROOT / "artifacts" / "models"
REPORT_ROOT = PROJECT_ROOT / "artifacts" / "reports"
REGISTRY_PATH = MODEL_ROOT / "registry.json"

In [2]:
registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
active_version = registry["active_model_version"]
active_entry = next(item for item in registry["models"] if item["model_version"] == active_version)
artifact_path = Path(active_entry["artifact_path"])
metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
checkpoint_path = Path(metadata["checkpoint_path"])
sequence_path = REPORT_ROOT / active_version / "lstm_sequences.npz"
scaler = ScalerStore.load(MODEL_ROOT / "scaler_params.joblib")

with np.load(sequence_path) as data:
    X_test = data["X_test"][:256]
    y_test = data["y_test"][:256]

print(f"Active model: {active_version}")
print(f"Calibration samples: {len(X_test)}")
print(f"Checkpoint: {checkpoint_path}")

Active model: lstm-real-20260518-032721
Calibration samples: 256
Checkpoint: C:\AIproject\AWAI\artifacts\models\lstm-real-20260518-032721\model.pt


In [3]:
@dataclass(frozen=True)
class MCUncertaintyResult:
    mean_kmh: np.ndarray
    std_kmh: np.ndarray
    lower_kmh: np.ndarray
    upper_kmh: np.ndarray
    confidence: np.ndarray
    latency_ms: float


class NotebookMCUncertaintyEstimator:
    def __init__(self, artifact_path: Path, scaler_store: ScalerStore, device: str = "cpu") -> None:
        self.artifact_path = artifact_path
        self.scaler_store = scaler_store
        self.device = torch.device(device)
        self.metadata = json.loads((artifact_path / "model_config.json").read_text(encoding="utf-8"))
        model_config = dict(self.metadata["model_config"])
        model_config["hidden_sizes"] = tuple(model_config["hidden_sizes"])
        self.model = TrafficLSTM(LSTMModelConfig(**model_config)).to(self.device)
        checkpoint = torch.load(self.metadata["checkpoint_path"], map_location=self.device, weights_only=False)
        self.model.load_state_dict(checkpoint["model_state_dict"])

    def predict(self, sequence_batch: np.ndarray, passes: int = 30, interval_z: float = 1.64) -> MCUncertaintyResult:
        if passes < 2:
            raise ValueError("MC Dropout requires at least 2 passes")
        started = time.perf_counter()
        tensor = torch.as_tensor(sequence_batch, dtype=torch.float32, device=self.device)
        samples = []
        self.model.train()
        with torch.no_grad():
            for _ in range(passes):
                scaled = self.model(tensor).detach().cpu().numpy()
                samples.append(self._inverse_speed(scaled))
        self.model.eval()
        stacked = np.stack(samples, axis=0)
        mean = np.mean(stacked, axis=0)
        std = np.std(stacked, axis=0)
        lower = np.clip(mean - interval_z * std, 0.0, 120.0)
        upper = np.clip(mean + interval_z * std, 0.0, 120.0)
        confidence = np.clip(1.0 - (std / 40.0), 0.05, 1.0)
        latency_ms = (time.perf_counter() - started) * 1000
        return MCUncertaintyResult(mean, std, lower, upper, confidence, latency_ms)

    def _inverse_speed(self, values: np.ndarray) -> np.ndarray:
        restored = self.scaler_store.inverse_transform_speed(values.reshape(-1, 1)).reshape(values.shape)
        return np.clip(restored, 0.0, 120.0)


estimator = NotebookMCUncertaintyEstimator(artifact_path=artifact_path, scaler_store=scaler)
result = estimator.predict(X_test, passes=30)
target_kmh = estimator._inverse_speed(y_test)
print("Mean prediction shape:", result.mean_kmh.shape)
print("Latency ms:", round(result.latency_ms, 3))

Mean prediction shape: (256, 4, 1)
Latency ms: 508.858


In [4]:
covered = (target_kmh >= result.lower_kmh) & (target_kmh <= result.upper_kmh)
coverage = float(np.mean(covered))
mean_interval_width = float(np.mean(result.upper_kmh - result.lower_kmh))
mean_std = float(np.mean(result.std_kmh))
mean_confidence = float(np.mean(result.confidence))
mae = float(np.mean(np.abs(result.mean_kmh - target_kmh)))
horizon_coverage = np.mean(covered.squeeze(-1), axis=0).round(6).tolist()
horizon_width = np.mean((result.upper_kmh - result.lower_kmh).squeeze(-1), axis=0).round(6).tolist()
single_summary = {
    "mean_kmh": result.mean_kmh[0].reshape(-1).round(3).tolist(),
    "lower_kmh": result.lower_kmh[0].reshape(-1).round(3).tolist(),
    "upper_kmh": result.upper_kmh[0].reshape(-1).round(3).tolist(),
    "confidence": result.confidence[0].reshape(-1).round(6).tolist(),
}

checks = {
    "shape_valid": result.mean_kmh.shape == y_test.shape,
    "finite_outputs": bool(np.isfinite(result.mean_kmh).all() and np.isfinite(result.std_kmh).all()),
    "interval_order_valid": bool((result.lower_kmh <= result.mean_kmh).all() and (result.mean_kmh <= result.upper_kmh).all()),
    "speed_bounds_valid": bool((result.lower_kmh >= 0).all() and (result.upper_kmh <= 120).all()),
    "confidence_bounds_valid": bool((result.confidence >= 0).all() and (result.confidence <= 1).all()),
    "calibration_sample_count_valid": len(X_test) == 256,
}
if not all(checks.values()):
    raise RuntimeError(checks)

report = {
    "status": "passed",
    "model_version": active_version,
    "runner_location": "notebooks/09_uncertainty_estimation_validation.ipynb",
    "method": "monte_carlo_dropout",
    "mc_passes": 30,
    "interval_z": 1.64,
    "calibration_samples": int(len(X_test)),
    "prediction_shape": list(result.mean_kmh.shape),
    "mean_prediction_kmh_first_sample": single_summary["mean_kmh"],
    "lower_bound_kmh_first_sample": single_summary["lower_kmh"],
    "upper_bound_kmh_first_sample": single_summary["upper_kmh"],
    "confidence_first_sample": single_summary["confidence"],
    "mean_confidence": mean_confidence,
    "mean_std_kmh": mean_std,
    "mean_interval_width_kmh": mean_interval_width,
    "mae_kmh": mae,
    "empirical_coverage": coverage,
    "horizon_coverage": horizon_coverage,
    "horizon_interval_width_kmh": horizon_width,
    "latency_ms": result.latency_ms,
    "checks": checks,
}

report_path = REPORT_ROOT / active_version / "uncertainty_estimation_validation.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps({"report": str(report_path), "coverage": coverage, "mean_confidence": mean_confidence, "mae": mae}, indent=2))

{
  "report": "C:\\AIproject\\AWAI\\artifacts\\reports\\lstm-real-20260518-032721\\uncertainty_estimation_validation.json",
  "coverage": 0.869140625,
  "mean_confidence": 0.951579749584198,
  "mae": 1.9274461269378662
}
